# 連接資料庫並取得評論資料

In [29]:
import os
import pandas as pd
from sqlalchemy import create_engine, text, bindparam

In [30]:
db_user = 'root'
db_password = '123456'
db_host = '100.77.42.49'
db_port = '3306'
db_name = 'lda_temp'

# 建立連線
connection_string = f"mysql+pymysql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}?charset=utf8mb4"
engine = create_engine(connection_string)

try:
    with engine.connect() as connection:
        result = connection.execute(text("SELECT 1"))
        print("連線成功！")
except Exception as e:
    print(f"連線失敗：{e}")

連線成功！


In [31]:
sql_file_path = 'sql_query/test.sql'
sql_file_name = os.path.splitext(os.path.basename(sql_file_path))[0]

In [32]:
try:
    # 讀取 SQL 檔案
    if not os.path.exists(sql_file_path):
        raise FileNotFoundError(f"找不到檔案：{sql_file_path}")

    with open(sql_file_path, 'r', encoding='utf-8-sig') as file:
        sql_query = file.read()

    # 執行查詢並放入 dataFrame
    with engine.connect() as conn:
        df = pd.read_sql(text(sql_query), conn)

    print(f"讀取成功！共取得 {len(df)} 筆資料")
    # print(df.head())

except FileNotFoundError as e:
    print(e)
except Exception as e:
    print(f"發生錯誤: {e}")

讀取成功！共取得 1986 筆資料


# 分詞

In [33]:
import pandas as pd
from gensim import corpora, models
from gensim.models.coherencemodel import CoherenceModel
import pyLDAvis
import pyLDAvis.gensim_models
import re
import os
import torch
import csv
from ckip_transformers.nlp import CkipWordSegmenter, CkipPosTagger, CkipNerChunker
import tqdm as notebook_tqdm
from tqdm.auto import tqdm

In [34]:
OUTPUT_PATH = f"result_Bertopic_0301"
STOP_WORDS_PATH = r"stop_dic/stopwords.txt"
CUSTOM_DICT_PATH = r'stop_dic/dict.txt'
POS_MAPPING_CSV_PATH = r"stop_dic/CKIP_Tag_Mapping_Table.csv"
DATA_PATH = r'data'
DEVICE = 0 if torch.cuda.is_available() else -1
print(f"{torch.cuda.is_available()}")

True


In [35]:
class CKIPTokenizer:
    """
    CKIPTokenizer 類別用於整合中研院 CKIP Transformers 斷詞工具
    """
    def __init__(self, stopwords_path, pos_mapping_csv_path, custom_dict_path=None, device=0):
        """
        初始化斷詞器與 CKIP 模型。
        :param stopwords_path: 停用詞檔案路徑 (.txt)
        :param pos_mapping_csv_path: 詞性對照表路徑 (.csv)，用於決定保留哪些詞性
        :param custom_dict_path: 自定義詞典路徑 (.txt)
        :param device: 執行設備 (0 為 GPU, -1 為 CPU)
        """
        self.stopwords = self.load_stopwords(stopwords_path)
        self.custom_dict = self.load_custom_dict(custom_dict_path) if custom_dict_path else {}
        self.pos_tags_to_keep = self.load_pos_tags_to_keep(pos_mapping_csv_path)
        
        # 初始化 CKIP 三大核心驅動器
        self.ws_driver  = CkipWordSegmenter (model="albert-base", device=device) # 斷詞
        self.pos_driver = CkipPosTagger     (model="albert-base", device=device) # 詞性標記
        self.ner_driver = CkipNerChunker    (model="albert-base", device=device) # 實體辨識
        
    def load_stopwords(self, filepath):
        """載入停用詞檔案，回傳一個 Set 以提升搜尋效率"""
        with open(filepath, encoding='utf-8-sig') as f:
            return set(line.strip() for line in f)

    def load_custom_dict(self, filepath):
        """載入自定義詞典，格式預期為：詞彙 [權重]"""
        custom_terms = {}
        with open(filepath, encoding='utf-8-sig') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 1:
                    word = parts[0]
                    # 若無指定權重，預設為 1.0
                    weight = float(parts[1]) if len(parts) > 1 else 1.0
                    custom_terms[word] = weight
        return custom_terms

    def load_pos_tags_to_keep(self, csv_path):
        """從 CSV 載入需要保留的詞性標記"""
        pos_tags = []
        with open(csv_path, newline='', encoding='utf-8-sig') as csvfile:
            reader = csv.DictReader(csvfile)
            for row in reader:
                # 檢查 CSV 中 '需要(0/1)' 欄位是否為 '1'
                if row['需要(0/1)'].strip() == '1':
                    tags = row['簡化標記']
                    pos_tags.append(tags)
        return pos_tags

    def tokenize(self, text, return_pos=False):
        """
        對單一字串進行斷詞
        :param text: 輸入純文字
        :param return_pos: 是否回傳 (詞, 詞性) 元組清單
        """
        if not isinstance(text, str):
            return ""

        # 清洗文字：僅保留中文 (Regex: \u4e00-\u9fff)
        text = re.sub(r'[^\u4e00-\u9fff]', '', text)
        if not text:
            return ""

        try:
            # 執行斷詞與詞性標記
            word_sentence_list = self.ws_driver([text], show_progress=False)
            pos_sentence_list = self.pos_driver(word_sentence_list, show_progress=False)

            words = word_sentence_list[0]
            pos_tags = pos_sentence_list[0]

            filtered_words = []
            filtered_word_pos = []
            
            # 過濾邏輯：1.不在停用詞內 2.字數>=2 3.屬於保留詞性
            for word, pos in zip(words, pos_tags):
                # if (word not in self.stopwords) and (len(word) >= 2) and (pos in self.pos_tags_to_keep):
                filtered_words.append(f"{word}")
                filtered_word_pos.append((word, pos))

            if return_pos:
                return filtered_word_pos
            else:
                return " ".join(filtered_words) # 回傳以空格分隔的字串
        except Exception as e:
            print(f"CKIP processing error: {e}")
            return "" if not return_pos else []
        
    def batch_tokenize(self, text_list, batch_size=64):
        """
        批次處理多筆文字
        :param text_list: 字串列表
        :param batch_size: 批次處理大小
        """
        clean_texts = []
        valid_indices = [] # 用於記錄非空值的位置，以便最後還原清單長度
        
        print("正在清理文字...")
        for idx, text in enumerate(text_list):
            if isinstance(text, str):
                # 這裡保留了中英數字 ( \w )
                # text = re.sub(r'[^\u4e00-\u9fff\w]', '', text) 
                text = re.sub(r'[^\u4e00-\u9fff]', '', text)
                if text.strip():
                    clean_texts.append(text)
                    valid_indices.append(idx)
        
        if not clean_texts:
            return [[] for _ in range(len(text_list))]

        # 批次執行模型運算
        print(f"執行 CKIP 模型 (Batch Size: {batch_size})...")
        ws_list = self.ws_driver(clean_texts, batch_size=batch_size, show_progress=True)
        pos_list = self.pos_driver(ws_list, batch_size=batch_size, show_progress=True)

        print("過濾停用詞與篩選詞性...")
        final_tokens_list = []
        
        for words, pos_tags in zip(ws_list, pos_list):
            filtered_words = []
            for word, pos in zip(words, pos_tags):
                # 條件篩選：非停用詞、長度 > 1、指定詞性
                if (word not in self.stopwords) and (len(word) > 1) and (pos in self.pos_tags_to_keep):
                    filtered_words.append(word)
            
            final_tokens_list.append(filtered_words)

        # 依照原始 list 的索引將結果放回對應位置，其餘填空 list
        result = [[] for _ in range(len(text_list))]
        for i, tokens in zip(valid_indices, final_tokens_list):
            result[i] = tokens
            
        return result

In [36]:
tokenizer = CKIPTokenizer(
    stopwords_path=STOP_WORDS_PATH, 
    pos_mapping_csv_path=POS_MAPPING_CSV_PATH, 
    custom_dict_path=CUSTOM_DICT_PATH, 
    device=DEVICE
)

In [37]:
print("處理 DataFrame...")
df['tokens'] = tokenizer.batch_tokenize(df['評論'].tolist(), batch_size=256)
print(df[['評論', 'tokens']].head())

處理 DataFrame...
正在清理文字...
執行 CKIP 模型 (Batch Size: 256)...


Inference:   0%|          | 0/8 [00:00<?, ?it/s]/root/miniconda3/envs/torch_222/lib/python3.10/site-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning:

The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:46.)

/root/miniconda3/envs/torch_222/lib/python3.10/site-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning:

The argument 'device' of Tensor.is_pinned() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:31.)

/root/miniconda3/envs/torch_222/lib/python3.10/site-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning:

The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:46.)

/root/miniconda3/envs/torch_222/lib/python3.10/site-packages/torch/utils/data/_util

過濾停用詞與篩選詞性...
                                                  評論  \
0  我的頭髮很多，而且不是那種很容易出油的髮質\n所以基本上不太挑洗髮精\n但不知道是不是今年天...   
1  去年9月所採購的洗髮品們，陸陸續續的使用，也即將用光光了～\n（因為＂不會＂再回購，所以在這...   
2  跟姊妹一起觀望一陣子，終於一起購入AVEDA洗髮精\n以前對洗髮這塊比較沒那麼重視\n總是以...   
3  每次洗完一瓶洗髮精都會換品牌，\n這個牌子是因為聽了podcast節目而認識的品牌，\n雖然...   
4  最近半年對頭髮的狀態實在是有點困擾\n本來就是細軟髮的髮質\n又因為冬季防冷戴各式帽子\n脫...   

                                              tokens  
0  [髮質, 基本, 今年, 天氣, 關係, 用到, 適合, 近期, 出門, 上班, 出遊, 戴...  
1  [去年, 採購, 洗髮品, 陸陸續續, 即將, 用光光, 回購, 空位, 用過, 淨髮, 精...  
2  [姊妹, 一起, 觀望, 一陣子, 終於, 一起, 購入, 以前, 那麼, 重視, 總是, ...  
3  [牌子, 節目, 認識, 藥妝店, 當時, 看起來, 好像, 不錯, 晚上, 痛苦, 頭皮,...  
4  [最近, 狀態, 實在, 困擾, 本來, 軟髮, 髮質, 冬季, 防冷, 帽子, 脫下, 處...  


In [38]:
file_name = f"result_tokens.csv"
save_path = os.path.join(OUTPUT_PATH, file_name)

print(f"\n準備儲存檔案至: {save_path}")

try:
    # 檢查目錄是否存在，不存在就建立
    if not os.path.exists(OUTPUT_PATH):
        os.makedirs(OUTPUT_PATH)
        print(f"已建立資料夾: {OUTPUT_PATH}")

    df[['評論', 'tokens']].to_csv(
        save_path, 
        index=False, 
        encoding='utf-8-sig'
    )
    
    print(f"成功保存，檔案大小約: {os.path.getsize(save_path) / 1024:.2f} KB")

except Exception as e:
    print(f"保存失敗: {e}")


準備儲存檔案至: result_Bertopic_0301/result_tokens.csv
已建立資料夾: result_Bertopic_0301
成功保存，檔案大小約: 2016.58 KB


# BERTopic

In [53]:
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance
from bertopic.vectorizers import ClassTfidfTransformer

In [54]:
# 將 tokens list 轉為空格分隔字串
df['token_str'] = df['tokens'].apply(lambda x: " ".join(x) if isinstance(x, list) else "")
initial_count = len(df)
df = df[df['token_str'].str.strip().astype(bool)].reset_index(drop=True)
docs = df['token_str'].tolist()

In [55]:
# 向量模型
# embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
embedding_model = SentenceTransformer("shibing624/text2vec-base-chinese")
embeddings = embedding_model.encode(docs, show_progress_bar=True)

Batches: 100%|██████████| 63/63 [00:02<00:00, 25.63it/s]


## 初始化模型

In [56]:
with open(STOP_WORDS_PATH, "r", encoding='utf-8-sig') as file:
    lines = file.readlines()
    stopwords = [line.strip() for line in lines]

vectorizer_model = CountVectorizer(
    stop_words=stopwords,
    # max_df=0.7,             # 如果一個詞在 N% 的文章都出現，就刪掉 (太通用)
    # min_df=1,               # 至少要出現 N 次才算數 (過濾錯字或極罕見詞)
    # ngram_range=(1, 2)      # 允許 "控油 效果" 這樣的雙詞組合，增加特徵豐富度
)


umap_model = UMAP(
    n_neighbors=10,     # 決定每個數據點在構建高維圖形時，要參考多少個鄰居
    n_components=5,     # 降維後的目標維度
    min_dist=0.0,      # 控制點、點之之間的最小距離
    metric='cosine',    # 距離度量方式
    random_state=0,
)

min_c_size = 10
hdbscan_model = HDBSCAN(
    min_cluster_size=min_c_size,        # 只有超過 N 筆相似的討論才會成一個 Topic
    min_samples=3,
    cluster_selection_epsilon=0.0,
    metric='euclidean',
    cluster_selection_method='eom', # leaf
    gen_min_span_tree=True
)

# c-TF-IDF
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)

# BERTopic
topic_model = BERTopic(
    language="multilingual",
    nr_topics=10,
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    min_topic_size=min_c_size,          # 與 HDBSCAN min_cluster_size 保持一致
    top_n_words=20,                     # 每個主題顯示多少個關鍵詞
    verbose=True,
)

## 訓練

In [57]:
print("Fitting BERTopic...")
topics, probs = topic_model.fit_transform(docs, embeddings=embeddings)

2026-03-01 15:15:31,510 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Fitting BERTopic...


2026-03-01 15:15:34,967 - BERTopic - Dimensionality - Completed ✓
2026-03-01 15:15:34,968 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-01 15:15:34,993 - BERTopic - Cluster - Completed ✓
2026-03-01 15:15:34,993 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-03-01 15:15:35,028 - BERTopic - Representation - Completed ✓
2026-03-01 15:15:35,028 - BERTopic - Topic reduction - Reducing number of topics
2026-03-01 15:15:35,029 - BERTopic - Topic reduction - Number of topics (10) is equal or higher than the clustered topics(6).
2026-03-01 15:15:35,029 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-01 15:15:35,157 - BERTopic - Representation - Completed ✓


## 查看主題結果

In [58]:
topic_info = topic_model.get_topic_info()
topic_info

,Topic,Count,Name,Representation,Representative_Docs
0,-1,1,-1_薑辣素_無解_強韌質_更加,"[薑辣素, 無解, 強韌質, 更加, 稀疏髮, 保濕力, 頭排水孔, 堅韌, 南投, 琥珀酸...",[採用 南投 生薑 內含 薑辣素 能夠 活絡 滋養 頭皮 強化 髮根 更加 強韌 彈性 添加...
1,0,10,0_齊腰_風情_俏麗_稱不上,"[齊腰, 風情, 俏麗, 稱不上, 粗介, 花容月貌, 時尚, 女孩, 飄逸, 經驗, 風格...",[稱不上 花容月貌 飄逸 秀髮 的確 齊腰 長髮 風情 俏麗 短髮 不同 時尚 風格 女孩 ...
2,1,21,1_年次_蓄長髮_大幅_漂過,"[年次, 蓄長髮, 大幅, 漂過, 下降, 毛病, 陸續, 長度, 毛糙, 連續, 隔天, ...",[說明 髮質 狀況 出油 細軟髮 蓄長髮 目前 剪短 長度 大概 洗頭 頭皮屑 連續 洗淨力...
3,2,1911,2_頭皮_味道_香味_起泡,"[頭皮, 味道, 香味, 起泡, 蓬鬆, 髮質, 透明, 控油, 乾澀, 清爽, 洗頭, 質...",[好用 舒緩 頭皮 泡泡 味道 舒服 順手 泡沫 綿密 深層 清潔 深入 髮芯 修護 避免 ...
4,3,14,3_生薑_薑汁_薑味_雪芙蘭,"[生薑, 薑汁, 薑味, 雪芙蘭, 強韌, 薑根, 油樺木, 抗屑, 美體, 白柳樹, 直播...",[薑汁 抗屑 只是 名子 天然 薑根 精華 油樺木 白柳樹 萃取 維他命 天然 蜂蜜 質地 ...
5,4,29,4_蘋果_生薑_防斷_蘋果味,"[蘋果, 生薑, 防斷, 蘋果味, 奇異果, 味道, 滋養, 清茶, 修護, 護髮素, 小勳...",[最近 入手 生薑 蘋果 防斷 滋養 評價 不錯 尤其 奇異果 嚇嚇 價錢 屈臣氏 打折 一...


In [59]:
try:
    save_path = os.path.join(OUTPUT_PATH, "topic_info.csv")
    topic_info.to_csv(save_path, index=False, encoding='utf-8-sig')
    print(f"\n成功保存至: {save_path}")
    print(f"文件大小: {os.path.getsize(save_path) / 1024 / 1024:.2f} MB")
except Exception as e:
    print(f"保存失敗: {e}")


成功保存至: result_Bertopic_0301/topic_info.csv
文件大小: 0.01 MB


## 保存評論與主題結果

In [60]:
doc_info = topic_model.get_document_info(docs)
df_reset = df.reset_index(drop=True)
doc_info_reset = doc_info.reset_index(drop=True)
topic_columns = doc_info_reset[['Topic', 'Name', 'Representation']]

final_df = pd.concat([df_reset, topic_columns], axis=1)
final_df['Keywords'] = final_df['Representation'].apply(lambda x: ", ".join(x) if isinstance(x, list) else str(x))
final_df.drop(columns=['Representation'], inplace=True)

output_filename = "reviews_with_topics.csv"
save_path = os.path.join(OUTPUT_PATH, output_filename)

try:
    final_df.to_csv(save_path, index=False, encoding='utf-8-sig')
    print(f"\n成功保存至: {save_path}")
    print(f"文件大小: {os.path.getsize(save_path) / 1024 / 1024:.2f} MB")
except Exception as e:
    print(f"保存失敗: {e}")


成功保存至: result_Bertopic_0301/reviews_with_topics.csv
文件大小: 3.76 MB


# 可視化

In [61]:
import plotly.io as pio
import plotly.express as px

pio.renderers.default = 'notebook'

### 條形圖

In [62]:
# C-TF-IDF 分數
fig = topic_model.visualize_barchart()
fig.write_html(os.path.join(OUTPUT_PATH, "bertopic_barchart.html"))
# fig.show()

In [63]:
# 查看指定主題下的c-TF-IDF 得分
topic_model.get_topic(0)

[('齊腰', 1.140864484090747),
 ('風情', 1.140864484090747),
 ('俏麗', 1.140864484090747),
 ('稱不上', 1.1258062445012482),
 ('粗介', 1.1258062445012482),
 ('花容月貌', 1.0981121313599047),
 ('時尚', 1.087710039226806),
 ('女孩', 1.076812928029043),
 ('飄逸', 1.002603367289227),
 ('經驗', 1.002603367289227),
 ('風格', 0.9961604543035677),
 ('尋覓', 0.9784572335937722),
 ('蜜糖', 0.9784572335937722),
 ('追求', 0.9782764439386221),
 ('中間', 0.9782764439386221),
 ('洗潤', 0.9511530691164944),
 ('的確', 0.9386329813390438),
 ('介紹', 0.918964671269914),
 ('用品', 0.9128401526786067),
 ('短髮', 0.8999916879744678)]

In [64]:
# topic_model.merge_topics(docs, [1, 3,17])
topic_info = topic_model.get_topic_info()
topic_info.to_csv(os.path.join(OUTPUT_PATH, "topic_info.csv"), index=False, encoding='utf-8-sig')

In [65]:
# 輸出主題資訊
doc_info = topic_model.get_document_info(docs)
doc_info.to_csv(os.path.join(OUTPUT_PATH, "document_info.csv"), index=False, encoding='utf-8-sig')

In [66]:
# 4. 保存主題分布 (Intertopic Distance Map)
fig_topics = topic_model.visualize_topics()
fig_topics.write_html(os.path.join(OUTPUT_PATH, "visualize_topics.html"))

# 5. 保存主題相似度熱力圖 (Heatmap)
fig_heatmap = topic_model.visualize_heatmap()
fig_heatmap.write_html(os.path.join(OUTPUT_PATH, "visualize_heatmap.html"))

# 6. 保存主題關鍵詞長條圖 (Barchart)
fig_barchart = topic_model.visualize_barchart(n_words=20)
fig_barchart.write_html(os.path.join(OUTPUT_PATH, "visualize_barchart.html"))

# 7. 保存階層可視化 (Hierarchy)
hierarchical_topics = topic_model.hierarchical_topics(docs)
fig_hierarchy = topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)
fig_hierarchy.write_html(os.path.join(OUTPUT_PATH, "visualize_hierarchy.html"))

# 8. 保存文檔散點可視化 (Documents / UMAP)
reduced_embeddings = UMAP(n_neighbors=10, n_components=2, min_dist=0.0, metric='cosine').fit_transform(embeddings)
fig_documents = topic_model.visualize_documents(docs, reduced_embeddings=reduced_embeddings)
fig_documents.write_html(os.path.join(OUTPUT_PATH, "visualize_documents.html"))

100%|██████████| 4/4 [00:00<00:00, 540.19it/s]
